#  **Cleaning The Datasheet:** `solar-eclipses.csv`

En este notebook Se realizara la limpieza del datasheet antes de comenzar de responder las preguntas **EDA**,
por lo que es necesario mantener la base de datos con datos limpios y no con datos crudos, Lo que busco es no afectar 
las respuestas **EDA**. Es por ello que comenzare hacer un checklist de los datos sucios detectados para posteriormente
comenzar con la limpieza de datos.

### Checklist Datos Crudos

| Columna   |       Accion        |
|-----------|---------------------|
| Date      | `Derivar la columna`|
| Duration  | `Derivar la columna y eliminar los nulos`|
| Region    | `Derivar la columna`|


- **Derivar la columna `Date`**: Lo veo realmente necesario derivar la columna **Date** ya que por lo que veo utilizare las derivadas
de tiempo para tener un mejor manejo de la informacion y poder asi crear columnas como: `day`,`month` y `year`.

- **Derivar la columna `Duration` y eliminar sus nulos**: Actualmente la columa `Duration` muestra los inutos y segundos en una misma columna,
por lo que la solucion optima que puedo hacer para la columna `Duration` es derivar la columna y crear dos nuevas columnas que sera `Duration-Min` y `Duration-Sec`,
me parece una forma excelente de representar asi las columnas nuevas, porque? por que esta haciendo una referencia directa a la columa `Duration`.

- **Columa `region`**: La columna `region`, es la columna con mas suciedad en e datasheet.

## **1.- Load File: `solar-eclipses.csv`**

In [187]:
import pandas as pd
pd.set_option('display.max_colwidth', 1000)
df = pd.read_csv('../data/solar-eclipses.csv')

## **2.- Cleaning Data**

### *1.- Derivando la columna `date`*

##### **Explicacion del codigo**

In [188]:
"""
CODE: 1.- Derivando la columna `date`

"""
df["day"]   = pd.to_datetime(df["date"]).dt.day     # Creando la columna "day" y asignando el numero del dia.
df["month"] = pd.to_datetime(df["date"]).dt.month   # Creando la columna "month" y asignando el numero del mes.
df["year"]  = pd.to_datetime(df["date"]).dt.year    # Creando la columna "year" y asignando el numero del año.

### *2.-Derivando la columna `duration`*

#### **Explicacion del codigo**
En la primera parte del codigo 







In [189]:
"""
CODE: 2.-Derivando la columna `duration` 

"""
# 1.- Reemplazando valores y separando los minutos y segundos en un nuevo dataframe
replace_duration = df["duration"].str.replace('s', '')
split_duration = replace_duration.str.split('m', expand=True)
split_duration = split_duration.astype('string')

df[["duration_min", "duration_sec"]] = split_duration

# Reemplazar NaN con 0
df["duration_min"] = df["duration_min"].fillna('0')
df["duration_sec"] = df["duration_sec"].fillna('0')

# Convertir a numérico
df["duration_min"] = pd.to_numeric(df["duration_min"], errors='coerce').fillna(0)
df["duration_sec"] = pd.to_numeric(df["duration_sec"], errors='coerce').fillna(0)

# Calcular la media redondeada (ignorando los 0 que eran NaN)
mean_min = round(df["duration_min"].replace(0, pd.NA).mean())
mean_sec = round(df["duration_sec"].replace(0, pd.NA).mean())

# Reemplazar los 0 con la media redondeada y convertir a entero
df["duration_min"] = df["duration_min"].replace(0, mean_min).astype(int)
df["duration_sec"] = df["duration_sec"].replace(0, mean_sec).astype(int)


# Agregando una nueva columna llamada "duration_total", Esta columna tiene todo el tiempo en segundos.
df["duration_total"] = df.eval("(60 * duration_min) + duration_sec") 

### *3.- Limpieza total de la columna `region`*

#### **Explicacion del codigo**

El objetivo del código es transformar la columna `region` desde un texto descriptivo y desordenado hacia una estructura limpia, consistente y analizable.

Primero se eliminan todas las anotaciones entre corchetes `[ ]`, ya que contienen información adicional del eclipse (tipo, trayectoria o comentarios editoriales) que no corresponde a regiones geográficas.

Después se normaliza el texto convirtiéndolo a minúsculas y eliminando espacios innecesarios para evitar duplicados causados por diferencias de formato.

Posteriormente se corrigen abreviaciones y variaciones de nombres geográficos (por ejemplo `N. America` → `north america`). Esto permite unificar categorías equivalentes.

Luego se separan las múltiples regiones que aparecen dentro de una misma celda utilizando comas como delimitador. Con `explode()` se transforma cada región en una fila independiente, lo cual es esencial para realizar conteos y análisis correctos.

Más adelante se convierten subregiones o países en su continente correspondiente mediante un diccionario de mapeo.

Finalmente se definen las regiones válidas (continentes) y se eliminan valores que no representan regiones reales, como océanos, polos o descripciones ambiguas.

El resultado es una columna `region` completamente normalizada donde:

- cada fila contiene una sola región
- todas las regiones pertenecen a un mismo nivel geográfico
- el dataset queda listo para visualización y análisis estadístico


In [190]:
"""
CODE: 3.- Limpieza total de la columna `region`

"""
# eliminar anotaciones entre corchetes
df['region'] = df['region'].str.replace(r"\[.*?\]", "", regex=True)

# normalizar texto
df['region'] = (
    df['region']
    .str.lower()
    .str.strip()
)

# normalizar abreviaciones
replace_dict = {
    "n. america": "north america",
    "s. america": "south america",
    "n america": "north america",
    "s america": "south america",
    "c. america": "central america",
    "ne asia": "asia",
    "nw asia": "asia",
    "se asia": "asia",
    "sw asia": "asia"
}

df['region'] = df['region'].replace(replace_dict, regex=True)

# separar múltiples regiones en listas
df['region'] = df['region'].str.split(",")

# crear una fila por región
df = df.explode('region')

# limpiar espacios restantes
df['region'] = df['region'].str.strip()

# mapear subregiones hacia continentes
continent_map = {
    "alaska": "north america",
    "central america": "north america",
    "madagascar": "africa",
    "new zealand": "australia",
    "east indies": "asia",
    "middle east": "asia"
}

df['region'] = df['region'].replace(continent_map)

# definir regiones válidas
valid_regions = [
    "africa",
    "asia",
    "europe",
    "north america",
    "south america",
    "australia",
    "antarctica"
]

# filtrar únicamente continentes válidos
df_clean = df[df['region'].isin(valid_regions)].copy()

# No es necesario tener la columna "duration".
df_clean = df_clean.drop(columns="duration")

## **3.- Eliminando columnas Inecesarias**

Eliminamos la columna `duration` ya que la funcionalidad que tiene en el datasheet es totalmente innecesaria ya que derivamos

su columna, por ello es que se ha decidido eliminarla por completo, El estar eliminando esta columna estaremos evitando mas procesamiento

y la duplicacion de informacion.

In [191]:
# df_clean = df_clean.drop(columns="duration")

## **4.- Exportando el datasheet `solar-eclipses-clean.csv`**

Una vez que hayamos detectado todas los datos crudos y despues de haber echo todo el procesamiento de datos

para obtener el datasheet sin ningun rastro de datos crudos. Ahora se procedera a exportar el datasheet `solar-eclipses-clean.csv`

con todos los datos limpios.

In [192]:
df_clean.to_csv("../data/solar-eclipses-clean.csv")